# ESTG ve GR Kozmolojik MCMC Analizi (Planck 2018 + BBN + LIGO O3)
Bu notebook, ESTG teorisinin serbest parametresi $\alpha_M$ ve standart kozmolojik parametreler $(\Omega_{\rm cdm}, \Omega_b, h)$ için Metropolis-Hastings MCMC zincirleri koşturur, Gelman-Rubin ($R-1$) yakınsama tanılarını hesaplar ve GetDist ile posterior dağılımlarını (Corner Plot) çizer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from getdist import plots, MCSamples

# 1. Likelihood ve Prior Fonksiyonları
def log_prior(theta):
    alpha_M, omega_cdm, omega_b, h = theta
    # Düz (flat) öncüller
    if not (-0.1 < alpha_M < 0.1 and 0.05 < omega_cdm < 0.25 and 0.015 < omega_b < 0.030 and 0.5 < h < 0.9):
        return -np.inf
    return 0.0

def log_likelihood(theta):
    alpha_M, omega_cdm, omega_b, h = theta
    # BBN (Cooke et al. 2018): Omega_b * h^2 = 0.0222 +- 0.0005
    obh2 = omega_b * h**2
    chi2_bbn = ((obh2 - 0.0222) / 0.0005)**2
    
    # Planck 2018 baseline kısıtları
    chi2_planck = ((alpha_M - 0.013) / 0.037)**2 + ((omega_cdm - 0.120) / 0.002)**2 + ((h - 0.674) / 0.005)**2
    
    # LIGO/Virgo O3 üst sınırı (|alpha_M| < 0.05)
    chi2_ligo = (alpha_M / 0.05)**2
    
    return -0.5 * (chi2_bbn + chi2_planck + chi2_ligo)

def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

# 2. Metropolis-Hastings MCMC Örnekleyicisi
def run_chain(n_steps=25000, start_pos=None):
    current = start_pos if start_pos is not None else np.array([0.0, 0.12, 0.022, 0.67])
    current_lp = log_posterior(current)
    prop_scale = np.array([0.025, 0.0015, 0.0004, 0.004])
    chain = np.zeros((n_steps, 4))
    accepted = 0
    
    for i in range(n_steps):
        proposal = current + np.random.normal(0, prop_scale)
        prop_lp = log_posterior(proposal)
        if np.log(np.random.uniform(0, 1)) < (prop_lp - current_lp):
            current = proposal
            current_lp = prop_lp
            accepted += 1
        chain[i] = current
    return chain, accepted / n_steps

# 4 Bağımsız Zincir Koştur
np.random.seed(42)
n_chains = 4
n_steps = 25000
burn_in = 5000

all_chains = []
for c_idx in range(n_chains):
    start = np.array([np.random.uniform(-0.02, 0.02), 0.12 + np.random.normal(0, 0.001), 0.022, 0.67])
    ch, acc = run_chain(n_steps, start)
    all_chains.append(ch[burn_in:])
    print(f'Zincir {c_idx+1} tamamlandı. Kabul Oranı: {acc:.2%}')

# 3. Gelman-Rubin Yakınsama Tanısı (R-1)
M = n_chains
N = n_steps - burn_in
chain_means = [np.mean(ch, axis=0) for ch in all_chains]
global_mean = np.mean(chain_means, axis=0)
B = (N / (M - 1)) * np.sum([(cm - global_mean)**2 for cm in chain_means], axis=0)
W = np.mean([np.var(ch, axis=0, ddof=1) for ch in all_chains], axis=0)
var_hat = ((N - 1) / N) * W + (1 / N) * B
R_minus_1 = np.sqrt(var_hat / W) - 1.0
print(f"Gelman-Rubin (R - 1) [alpha_M]: {R_minus_1[0]:.4f} (Hedef: < 0.01)")

# 4. GetDist ile Corner Plot Çizimi
merged_samples = np.vstack(all_chains)
names = ['alpha_M', 'Omega_cdm', 'Omega_b', 'h']
labels = [r'\alpha_M', r'\Omega_{\rm cdm}', r'\Omega_b', r'h']

mcsamples = MCSamples(samples=merged_samples, names=names, labels=labels)
g = plots.get_subplot_plotter()
g.triangle_plot(mcsamples, filled=True, contour_colors=['#1f77b4'])
plt.savefig('corner_plot.png', dpi=200, bbox_inches='tight')
plt.show()

alpha_vals = merged_samples[:, 0]
print(f"alpha_M Ortalama: {np.mean(alpha_vals):.4f} +- {np.std(alpha_vals):.4f}")
print(f"95% Guven Araligi: |alpha_M| < {np.percentile(np.abs(alpha_vals), 95):.4f}")
np.savetxt('mcmc_chain_merged.txt', merged_samples, header='alpha_M Omega_cdm Omega_b h')